# ElementFlow — Security Audit & Architecture Review

**Scope:** `contracts/` in the ElementFlow repository, read against its consumer, the
`element-pay-aggregator` FastAPI backend.
**Live deployments reviewed:** Base mainnet and Base Sepolia UUPS proxies recorded in `.openzeppelin/`.
**Deliverable status:** audit complete, fixes implemented, 113 tests passing.

---

## 1. Executive summary

ElementFlow escrows and settles fiat on-ramp and off-ramp orders. The v1 contract
(`OrderManagement`) is live behind UUPS proxies on Base and Base Sepolia and is driven by
a Python backend that submits `createOrder`, `settleOrder` and `refundOrder` transactions.

The audit found **four critical and five high-severity issues**, two of which are
exploitable today by any address with no special access:

| # | Severity | Issue |
|---|----------|-------|
| C-1 | Critical | `escrowFunds`/`releaseEscrow` let **anyone** permanently freeze another user's escrow |
| C-2 | Critical | `createOrder` has **no access control** — anyone can spend any approved allowance |
| C-3 | Critical | On-ramp settlement can pay out of **off-ramp user escrow** (funds are not segregated) |
| C-4 | Critical | `require(token.transfer(...))` **bricks the contract for USDT** and similar tokens |
| H-1 | High | Order IDs derive from `block.timestamp` — collision-prone and front-runnable |
| H-2 | High | On-ramp orders can **never be refunded**; failed orders stay `Pending` forever |
| H-3 | High | No reentrancy guard anywhere, despite the README claiming otherwise |
| H-4 | High | Concurrent on-ramp orders **oversubscribe** the same liquidity |
| H-5 | High | `settleOrder` is `payable` with no withdrawal path — any ETH sent is **permanently lost** |

The response is a storage-compatible v2 implementation, `ElementFlowOrderManager`, that
**upgrades the existing proxies in place**. The address does not change, pending orders
survive, and the backend keeps working against the same ABI. This matters: C-1 and C-2 are
live, and an in-place upgrade closes them in one transaction rather than requiring a user
migration during which the holes stay open.

v2 also introduces a **provider/adapter abstraction** so that a Yellow Card-style partner
can be added as a settlement route without touching the contract that custodies user
escrow. ElementFlow's own treasury is implemented as the first adapter, which keeps the
abstraction honest — the default path exercises the same interface a partner will.

> **One thing to decide before deploying.** The `LEGACY_ESCROW` seed described in §6 is a
> manual input to the upgrade. Getting it wrong in the unsafe direction lets v2 spend live
> user escrow as house float. `scripts/scan-legacy-orders.js` computes it; §6 explains how
> to verify it.

---

## 2. Architecture review

### 2.1 What v1 actually is

```
                 ┌──────────────────────────────┐
   backend ─────▶│  OrderManagement (UUPS)      │
  (one hot key)  │                              │
                 │  • orders mapping            │
                 │  • holds ALL tokens:         │
                 │      off-ramp escrow         │
                 │      + on-ramp float         │  ← conflated (C-3)
                 │      + accidental transfers  │
                 │  • Ownable, single owner     │
                 │  • no reentrancy guard       │
                 │  • escrowFunds / releaseEscrow: dead code that
                 │    mutates state (C-1)       │
                 └──────────────────────────────┘
```

Three structural problems, independent of any individual bug:

1. **No separation of funds.** One balance serves user escrow, house liquidity and stray
   transfers. Every accounting question ("can we pay this out?") is answered with
   `balanceOf(this)`, which is true of money that is already spoken for.
2. **No separation of duties.** `Ownable` gives one key upgrade rights, config rights and
   pause rights; a separate hardcoded `_aggregatorAddress` gets settlement. A compromise of
   either is total within its domain, and the owner key is also the upgrade key.
3. **Settlement policy is hardcoded.** On-ramp pays from the contract's own balance. There
   is no seam at which a partner, an OTC desk, or a second treasury could be introduced
   without rewriting `settleOrder`.

`SettingsManager` (`OrderManagerSetting.sol`) was a token allowlist that **nothing
referenced** — it was never deployed (the `.openzeppelin` manifests contain only
`OrderManagement` implementations) and its `initialize()` had no `_disableInitializers()`
constructor guard. It has been deleted; the allowlist is now enforced inside the manager,
where it actually gates order creation.

### 2.2 v2 target architecture

```
                    ┌───────────────────────┐
                    │  ProviderRegistry     │   providerId ──▶ adapter
                    │  (UUPS, own roles)    │   enable / disable / swap
                    └───────────┬───────────┘
                                │ resolves
                                ▼
  backend ──▶ ┌──────────────────────────────────────┐
  (AGGREGATOR │  ElementFlowOrderManager (UUPS)      │
   _ROLE)     │                                      │
              │  escrowedBalance[token]  ← user      │  segregated
              │  reservedLiquidity[token] ← house    │  accounting
              │  balance ≥ escrowed + reserved       │  (invariant)
              │                                      │
              │  roles: ADMIN / UPGRADER / PAUSER    │
              │         AGGREGATOR / ORDER_CREATOR   │
              │         TREASURER                    │
              └───────────────┬──────────────────────┘
                              │ IOnRampProvider
             ┌────────────────┴─────────────────┐
             ▼                                  ▼
   ┌──────────────────┐              ┌──────────────────────┐
   │  TreasuryPool    │              │  Partner adapter     │
   │  (in-house, UUPS)│              │  (Yellow Card-style) │
   │  default route   │              │  added, not built in │
   └──────────────────┘              └──────────────────────┘
```

The split is deliberate. Onboarding or cutting off a settlement partner is an **operational**
action that should never require upgrading the contract holding user escrow; those are
different contracts with different roles and different change cadences.

---

## 3. Security findings and fixes

Each finding below has a corresponding regression test. The v1 exploits are reproduced
against the original v1 source in `test/Upgrades.test.js` so the fixes are demonstrated,
not merely asserted.

### C-1 — Anyone can permanently freeze another user's escrow

**v1 code:**

```solidity
function escrowFunds(bytes32 _orderId, uint256 _amount) external whenNotPaused {
    Order storage order = orders[_orderId];
    require(order.status == OrderStatus.Pending, "Order is not pending");
    require(order.amount >= _amount, "Escrow amount exceeds order amount");
    order.provider = msg.sender;          // ← no access control, no funds moved
    emit EscrowReleased(_orderId);
}

function releaseEscrow(bytes32 _orderId) external whenNotPaused {
    Order storage order = orders[_orderId];
    require(order.provider == msg.sender, "Caller is not the escrow provider");
    require(order.status == OrderStatus.Pending, "Order is not pending");
    order.status = OrderStatus.Completed;  // ← terminal, no tokens transferred
    emit EscrowReleased(_orderId);
}
```

**Exploit.** For any pending off-ramp order, an attacker calls `escrowFunds(orderId, 0)`
(passes: `order.amount >= 0`), which makes them the "provider". They then call
`releaseEscrow(orderId)`, moving the order to `Completed` **without transferring
anything**. `settleOrder` and `refundOrder` both require `Pending`, so the user's tokens
are now stranded in the contract with no code path that can release them. Cost to the
attacker: two transactions. Damage: the full order amount, permanently, per order.

**Fix.** Both functions are removed. Escrow is established by the token transfer inside
`createOrder`, and the only terminal transitions are `settleOrder` and `refundOrder`, both
of which move funds. There is no longer any way to reach a terminal state without a
transfer.

### C-2 — `createOrder` is completely permissionless

v1's `createOrder` accepts an arbitrary `_userAddress` and pulls tokens from it. Any user
who has approved the contract (which every off-ramp user must) can have that allowance
spent by anyone, at any amount, at any time.

**Fix.** Creation now requires either that the caller *is* the requester, or that it holds
`ORDER_CREATOR_ROLE`:

```solidity
if (msg.sender != requester && !hasRole(ORDER_CREATOR_ROLE, msg.sender)) {
    revert NotOrderCreator(msg.sender, requester);
}
```

This preserves the production flow — the backend relays orders using a dedicated key, which
now holds `ORDER_CREATOR_ROLE` — while also supporting direct self-service creation from a
dApp. It additionally closes an order-ID griefing vector: an attacker can no longer
pre-create an order at a victim's ID to block it.

### C-3 — On-ramp settlement can drain off-ramp escrow

v1 answers every solvency question with `IERC20(token).balanceOf(address(this))`. That
balance includes off-ramp escrow belonging to users with pending orders. An on-ramp order
therefore passes its liquidity check against money it does not own, and settles by paying it
out. The off-ramp user's later refund then fails for lack of funds.

**Fix.** Funds are segregated by purpose and the contract maintains an explicit invariant:

```
balanceOf(this) ≥ escrowedBalance[token] + reservedLiquidity[token]
```

`availableLiquidity()` returns only the surplus above both, and it is the sole basis for
on-ramp reservations, `withdrawLiquidity` and `rescueTokens`. A test walks a mixed workload
of overlapping on-ramp and off-ramp orders and asserts the invariant after every state
change.

### C-4 — `require(token.transfer(...))` bricks the contract for USDT

USDT and several other major tokens do not return a `bool` from `transfer`/`transferFrom`.
Solidity's ABI decoder reverts when a return value is declared but absent, so **every v1
operation on such a token reverts** — orders cannot be created, settled or refunded. For a
payments system whose primary settlement asset class is stablecoins, this rules out a large
part of the intended market.

**Fix.** All transfers go through OpenZeppelin `SafeERC20`. `test/Security.test.js` runs a
full off-ramp lifecycle against a no-return mock, and `test/Upgrades.test.js` confirms the
same flow reverts on v1.

### H-1 — Order IDs derive from `block.timestamp`

```solidity
orderId = keccak256(abi.encodePacked(block.timestamp, _userAddress, _amount, _token));
```

Two consequences. **Collision:** a user placing two identical orders in the same block gets
the same ID, and the second reverts on `require(orders[orderId].requester == address(0))` —
a legitimate order fails for no good reason. **Predictability:** the ID depends only on
public inputs, so it can be computed ahead of time and, combined with C-2, pre-created to
block a specific order.

This is also the on-chain half of **BUG-01** in the aggregator (`docs/local/BUG-01_*`), the
off-ramp double-payment incident. That investigation concluded the on-chain `order_id` was
the wrong idempotency boundary because two identical customer intents produced two distinct
valid orders. v1's ID scheme makes that unavoidable: it has no notion of intent.

**Fix.** IDs are derived from an explicit caller-supplied `intentKey`, domain-separated by
chain and contract:

```solidity
keccak256(abi.encode(block.chainid, address(this), requester, amount, token, orderType, intentKey))
```

This gives the contract **true idempotency**: replaying an intent reverts with
`OrderAlreadyExists` rather than creating a second payable order, which turns BUG-01's
failure mode into a contract-level guarantee instead of a backend responsibility. Two
genuinely distinct orders with identical parameters remain possible via distinct intent
keys, including within one block. The v1-compatible `createOrder` derives the key from the
message hash, which the backend already generates uniquely per intent, so no backend change
is needed to get the benefit.

### H-2 — On-ramp orders can never be refunded

`refundOrder` has `require(order.orderType == OrderType.OffRamp)`. A failed on-ramp order —
the user's M-Pesa payment is declined, say — stays `Pending` forever. Under v2's accounting
that would also permanently strand the reserved float.

**Fix.** `refundOrder` handles both directions. An on-ramp refund releases the reservation
(no tokens were ever escrowed on-chain, since the user pays fiat); an off-ramp refund
returns the full escrowed amount.

### H-3 — No reentrancy protection

v1 has no guard on any function, and the README's claim of "Reentrancy Guards: Protected
against reentrancy attacks" is simply false. v1 also violates checks-effects-interactions in
`refundOrder`, which transfers **before** setting `order.status = Cancelled`.

**Fix.** `ReentrancyGuardUpgradeable` on every state-changing entry point, and strict CEI
throughout — status is always written before any external call. Tests use an ERC777-style
callback token to re-enter during both settlement and creation, and assert both that the
call reverts and that escrow accounting is unchanged afterwards.

### H-4 — Concurrent on-ramp orders oversubscribe liquidity

v1 checks `balanceOf(this) >= _amount` at creation but reserves nothing. Ten concurrent
orders each pass against the same balance; settlement then fails for whichever arrive after
the money runs out — after the users have already paid fiat.

**Fix.** `reservedLiquidity[token]` is committed at creation and released on settle/refund.
A test creates two 15k orders against 20k of float and asserts the second is rejected.

### H-5 — `settleOrder` is `payable` with no withdrawal path

v1's `settleOrder` accepts ETH and the contract has no `withdraw`, no `receive`, and no
rescue function. Any ETH sent is permanently lost.

**Fix.** No function is payable and no `receive`/`fallback` exists, so ETH transfers revert
outright. ERC20s sent by mistake are recoverable via `rescueTokens`, bounded by
`availableLiquidity` so a rescue can never reach user escrow.

### Medium and informational findings

| ID | Severity | Finding | Fix |
|----|----------|---------|-----|
| M-1 | Medium | `approveTokensForContract` calls `token.approve(address(this), ...)` — the contract approves *itself*, which is meaningless and misleads integrators into thinking it grants an allowance | Removed |
| M-2 | Medium | No token allowlist enforced; any ERC20 (including fee-on-transfer and rebasing) could be escrowed and silently break accounting | `isTokenAllowed` gate on creation, plus an exact-balance-delta check that rejects fee-on-transfer tokens |
| M-3 | Medium | Single-step `Ownable`; a typo'd `transferOwnership` bricks upgrades permanently | `AccessControl` with multiple admins possible; role grants are reversible |
| M-4 | Medium | No order expiry — a stalled or censoring aggregator traps user escrow indefinitely with no user recourse | `refundExpiredOrder` is permissionless after TTL and always pays the requester |
| M-5 | Medium | `MAX_BPS` declared but never used; no fee mechanism, and the `rate` field in `OrderCreated` is hardcoded to `0` | Fee in bps with a `MAX_FEE_BPS` cap, snapshotted per order |
| M-6 | Medium | Dead `SettingsManager` with an unguarded `initialize()` on the implementation | Deleted; allowlist folded into the manager |
| L-1 | Low | `getOrder` reverts on unknown IDs while `orders()` returns a zero struct — two different "not found" behaviours | Single `OrderStatus.None` sentinel; `getOrder` reverts, `getOrderRecord` returns the raw struct |
| L-2 | Low | Duplicate `IOrderManagement.sol` at two paths with diverging signatures | Stale copy deleted |
| L-3 | Low | Stray second `hardhat.config.js` under `scripts/` | Deleted |
| L-4 | Low | No events on `updateTreasury` / `updateAggregatorAddress`, so config drift is invisible to monitoring | Every setter emits a before/after event |
| L-5 | Info | README claims reentrancy guards and comprehensive validation that do not exist | Claims now match the code |

### Accepted risks

These are deliberate trade-offs rather than oversights, and each is bounded:

- **Expiry front-running.** Once an order passes its TTL, anyone may refund it — including
  moments before the aggregator settles. `ORDER_TTL` must exceed the settlement SLA with
  margin. The alternative (aggregator-only refunds) reintroduces the fund-trapping risk of
  M-4, which is strictly worse.
- **Provider fee delivery is not balance-verified.** The manager verifies the *beneficiary's*
  balance delta on provider-routed on-ramp settlement, but not the fee recipient's. A
  misbehaving adapter could shortchange ElementFlow's own revenue; it cannot shortchange a
  user. Verifying both would break when `requester == feeRecipient`.
- **Admin is trusted for configuration.** `DEFAULT_ADMIN_ROLE` can retarget the treasury and
  fee recipient and can grant itself any role. The fee is capped at `MAX_FEE_BPS` in code, and
  no admin action can reach escrowed user funds — `withdrawLiquidity` and `rescueTokens` are
  both bounded by `availableLiquidity`. Governance should sit behind a multisig + timelock.
- **Tokens that turn malicious after allowlisting.** An upgradeable token could add a
  transfer fee after being allowlisted, under-delivering on settlement. Allowlisting is a
  deliberate governance act and should favour immutable or well-governed tokens.

---

## 4. Upgradeability plan

### 4.1 Why in-place, not a redeploy

The natural instinct with a rewrite this size is to deploy fresh and migrate. That is the
wrong call here, for a specific reason: **C-1 and C-2 are live**. A migration leaves the
vulnerable proxy accepting orders for as long as it takes users and integrators to move,
and every pending order during that window is exposed. An in-place upgrade closes both in a
single transaction and keeps the address stable for the backend, the frontend and the
already-deployed contract address map in `CONTRACT_ADDRESS_MAP`.

This is only possible because the layout works out. v2 reproduces v1's first three slots
exactly:

| Slot | v1 | v2 | Note |
|------|----|----|------|
| 0 | `address _aggregatorAddress` | `address _legacyAggregatorAddress` | `@custom:oz-renamed-from` |
| 1 | `address treasury` | `address treasury` | value carried over, not overwritten |
| 2 | `mapping(bytes32 => Order) orders` | `mapping(bytes32 => LegacyOrder) legacyOrders` | struct layout identical |

All v1 parent contracts (`Ownable`, `Pausable`, `UUPS`) use OpenZeppelin v5 **namespaced
ERC-7201 storage**, so replacing `OwnableUpgradeable` with `AccessControlUpgradeable`
disturbs nothing above. v2 state is appended after slot 2, followed by a `uint256[38] __gap`.

One subtlety the OpenZeppelin validator caught during development and which is worth
recording: dropping `OwnableUpgradeable` **deletes its ERC-7201 namespace**, which the
validator rejects. v2 therefore retains an inert `DeprecatedOwnableStorage` struct carrying
the `@custom:storage-location erc7201:openzeppelin.storage.Ownable` annotation. Without it,
a future version could unknowingly reuse slots that still hold the v1 owner value.

### 4.2 Initialization versioning

Both entry points are `reinitializer(2)`, which makes them mutually exclusive:

- `initialize(...)` — fresh proxies. Jumping straight to version 2 means `initializeV2` can
  never be called on a new deployment.
- `initializeV2(...)` — live v1 proxies, which sit at version 1.

Tests assert that neither can be re-run and that a proxy migrated with one rejects the other.

### 4.3 A footgun worth naming

Parent `__X_init()` functions are `onlyInitializing`, which means a `reinitializer` **may**
legally call them — and `__Pausable_init()` sets `_paused = false`. Re-running it during an
upgrade would **silently unpause a contract that governance had deliberately halted**,
which is precisely when you least want the system to restart itself. Neither `initializeV2`
nor the v3 example calls it. `ElementFlowOrderManagerV3Mock` documents this inline as a
template for future upgrades.

### 4.4 Upgrade authority

`_authorizeUpgrade` is gated on `UPGRADER_ROLE`, held separately from `AGGREGATOR_ROLE`. The
backend's hot settlement key cannot upgrade the contract — a distinction v1 did not draw,
where the single owner key held everything.

---

## 5. On-ramp provider abstraction

### 5.1 The requirement

Today ElementFlow settles on-ramps from its own liquidity. The system needs to support
partner-supplied liquidity — a Yellow Card-style integration — **without rewriting the
contract that holds user escrow**. The aggregator already carries partner scaffolding
(`app/utils/partner_yellowcard_utils.py`, `partner_yc_preflight.py`, `partner_yc_limits.py`),
so the on-chain seam is the missing half.

### 5.2 The seam

Every order carries a `providerId`. `bytes32(0)` means internal liquidity; anything else
resolves through `ProviderRegistry` to an adapter implementing `IOnRampProvider`:

```solidity
interface IOnRampProvider {
    function providerId() external view returns (bytes32);
    function isTokenSupported(address token) external view returns (bool);
    function availableLiquidity(address token) external view returns (uint256);

    function settleOnRamp(OrderContext calldata ctx, address beneficiary, address feeRecipient) external;
    function settleOffRamp(OrderContext calldata ctx) external;

    function onOrderCreated(OrderContext calldata ctx) external;   // pre-flight / reserve
    function onOrderRefunded(OrderContext calldata ctx) external;  // release
}
```

Adding a partner is then: deploy an adapter, `registerProvider(id, adapter)`, and either
route specific orders to it via `createOrderWithProvider` or flip `setDefaultProviderId`.
**No change to the order manager.** Routes can be disabled independently by a guardian role
without a governance round-trip.

### 5.3 The trust model — the part that matters

An adapter is a third party. The design assumes it may be buggy, compromised, or dishonest.

**The manager never trusts an adapter's word that it paid.** On provider-routed on-ramp
settlement it measures the beneficiary's balance across the call:

```solidity
uint256 balanceBefore = IERC20(token).balanceOf(requester);
IOnRampProvider(adapter).settleOnRamp(ctx, requester, feeRecipient);
uint256 delivered = IERC20(token).balanceOf(requester) - balanceBefore;
if (delivered < netAmount) revert ProviderSettlementShortfall(netAmount, delivered);
```

An adapter that underpays, pays the wrong address, or silently no-ops **cannot mark an order
settled** — it can only fail closed, leaving the order `Pending` and refundable. Three tests
drive exactly these three behaviours against a configurable hostile adapter.

Corollary: adapters **push** funds. The manager holds no allowance on them and makes no
assumption about where their liquidity comes from, so a compromised adapter has no standing
claim on manager-held funds.

For off-ramp the direction reverses — the manager funds the adapter *before* asking it to run
the fiat leg, so a partner is never expected to front the payout from its own balance sheet.

### 5.4 Keeping the abstraction honest

`TreasuryPool` — ElementFlow's own liquidity — is itself an `IOnRampProvider` and the default
route. This is a deliberate choice: the everyday path exercises the same interface a partner
will, so the abstraction cannot quietly rot into something that only works for the in-house
case. It also moves house float out of the escrow contract, so the two have separate
custody and separate roles.

---

## 6. Deployment and upgrade guidance

### 6.1 Fresh deployment

```bash
cp .env.example .env      # then fill in the four role addresses
npx hardhat test          # 113 tests must pass
npm run deploy -- --network base-sepolia
```

Roles **must** be four distinct addresses in production. The script warns loudly and falls
back to the deployer for any that are unset — fine locally, unacceptable on mainnet.

| Role | Holder | Why |
|------|--------|-----|
| `DEFAULT_ADMIN_ROLE` | Governance multisig (ideally + timelock) | Config, token allowlist, role grants |
| `UPGRADER_ROLE` | Same multisig | Kept apart from settlement so a hot-key compromise cannot upgrade |
| `PAUSER_ROLE` | Multisig **and** an on-call hot key | Pausing is fail-safe, so it should be frictionless |
| `AGGREGATOR_ROLE` / `ORDER_CREATOR_ROLE` | Backend settlement key | Hot by necessity; cannot upgrade, pause or move liquidity |
| `TREASURER_ROLE` | Treasury multisig | Float in/out, bounded by `availableLiquidity` |

Unpausing is intentionally **admin-only** while pausing is broader: halting the system is a
safety action, restarting it is a governance decision.

### 6.2 Upgrading the live v1 proxies

The escrow seed is the one genuinely dangerous input, so do this in order:

```bash
# 1. Compute the seed. Uses on-chain status, not event replay — v1's releaseEscrow
#    marks orders Completed without emitting OrderSettled.
PROXY_ADDRESS=0x8B5B... FROM_BLOCK=<deployment block> \
  npx hardhat run scripts/scan-legacy-orders.js --network base

# 2. Sanity-check the output: for each token, escrow must be ≤ the proxy's balance.
#    Seeding too LOW lets v2 spend live user escrow as house float.
#    Seeding too HIGH only locks house float, which a treasurer can release later.
#    When in doubt, round up.

# 3. Upgrade. Validates storage layout before broadcasting anything.
PROXY_ADDRESS=0x8B5B... LEGACY_ESCROW=0xTokenA:1000000,0xTokenB:5000000 \
  npx hardhat run scripts/upgrade-v1-to-v2.js --network base
```

The upgrade must be signed by the current v1 `owner`, since v1 gates `_authorizeUpgrade`
with `onlyOwner`. The script checks this before doing anything.

**Post-upgrade, in order:**

1. `setTokenAllowed(token, true)` for every supported token — **v2 rejects unlisted tokens,
   so order creation stays down until this is done.**
2. Deploy `ProviderRegistry` + `TreasuryPool`, `registerProvider`, `setProviderRegistry`.
3. Drain pending v1 orders with `settleLegacyOrder` / `refundLegacyOrder`; watch
   `escrowedBalance` fall to zero.
4. Transfer `DEFAULT_ADMIN_ROLE` and `UPGRADER_ROLE` to the multisig, then renounce the
   deployer's roles.

### 6.3 Backend compatibility

The v2 ABI is deliberately backward-compatible on the paths the aggregator uses. One detail
is load-bearing and worth stating plainly:

`transaction_controller.py` decodes `getOrder` **positionally** and reads `order_data[5]` as
`0=PENDING, 1=SETTLED, 2=REFUNDED`. v2's internal enum is
`None=0, Pending=1, Settled=2, Refunded=3`. Returning the raw enum would make the backend
read **every pending order as already settled** — a silent, high-severity integration
failure. `getOrder` therefore preserves the exact flat 8-tuple *and* re-encodes the status
into v1 numbering; `getOrderRecord` exposes the true v2 struct for new code. A test asserts
the v1 encoding for all three states.

Also preserved: `createOrder(address,uint256,address,uint8,string)`, `settleOrder(bytes32)`,
`refundOrder(bytes32)`, `checkAllowance(address,address)`, and the `OrderCreated` event
signature including its now-unused trailing `rate` field.

**Backend changes required:**

- Handle custom-error reverts. v2 uses typed errors (`OrderAlreadyExists`,
  `InsufficientLiquidity`, `NotOrderCreator`, …) rather than revert strings; existing string
  matching in the retry logic needs updating.
- Treat `OrderAlreadyExists` as **success-idempotent**, not failure — it means the intent was
  already recorded. This is the contract-level fix for the BUG-01 double-payment class.
- The backend's create key needs `ORDER_CREATOR_ROLE` and its settle key `AGGREGATOR_ROLE`
  (the same address may hold both).

---

## 7. Test strategy

`npx hardhat test` — **113 tests, all passing.**

| Suite | Tests | Focus |
|-------|-------|-------|
| `OrderLifecycle.test.js` | 29 | Happy paths both directions, ID derivation and idempotency, state-transition legality, expiry, input validation, backend ABI compatibility |
| `Security.test.js` | 36 | Access control per role, pause semantics, reentrancy, hostile/non-standard tokens, fee arithmetic, liquidity safety |
| `ProviderFlows.test.js` | 26 | Registry lifecycle, provider-routed settlement, adversarial adapters, `TreasuryPool` |
| `Upgrades.test.js` | 22 | v1 exploit reproduction, in-place v1→v2 upgrade, layout validation, v2→v3 |

The strategy rests on four ideas:

**1. Exploit the old contract, then prove the fix.** `Upgrades.test.js` deploys the real v1
source and *actually executes* the C-1 freeze, the C-2 allowance theft, the H-2 refund gap
and the C-4 USDT revert. These are regression tests in the strict sense: they would pass
against v1 and fail if v2 ever reintroduced the behaviour.

**2. Assume every external contract is hostile.** Mocks cover the ways real tokens and real
partners misbehave: no-return (USDT), fee-on-transfer, transfers that return `false`,
transfers that revert, and an ERC777-style reentrancy callback. `MockOnRampProvider` has
switches for underpaying, paying the wrong address, paying nothing, and reverting in each
hook.

**3. Assert invariants, not just outcomes.** The mixed-workload test re-checks
`balance ≥ escrowed + reserved` after **every** state change across interleaved on-ramp and
off-ramp orders, rather than checking a final balance. Failure-path tests assert that
accounting is *unchanged* after a revert, not merely that the call failed.

**4. Let the upgrade validator do real work.** Layout compatibility is asserted positively
(v1→v2 validates) and negatively (a reordered layout and a non-UUPS implementation are both
rejected). This is what caught the deleted-Ownable-namespace issue in §4.1.

### Recommended before mainnet

- `npx hardhat coverage`, targeting >95% on `contracts/` excluding mocks.
- Fuzz/invariant testing (Foundry or Echidna) on the accounting invariant, with a random
  sequence of create/settle/refund across both order types.
- Slither and Mythril in CI.
- An external audit of the final `ElementFlowOrderManager` + `TreasuryPool` pair.
- A Base Sepolia rehearsal of the full §6.2 upgrade runbook, including the escrow seed, and
  a dry run against a mainnet fork.

---

## 8. Appendix — running the suite

The cells below are the exact commands used to validate this work. They are shell cells:
run them from the repository root with a Bash-capable Jupyter kernel, or copy them into a
terminal.

In [ ]:
!npx hardhat compile

In [ ]:
!npx hardhat test

In [ ]:
# Storage-layout compatibility between the live v1 implementation and v2,
# checked without broadcasting anything.
!npx hardhat test test/Upgrades.test.js --grep "storage layout validation"

In [ ]:
# The v1 exploits, reproduced against the original source.
!npx hardhat test test/Upgrades.test.js --grep "v1 vulnerabilities"

In [ ]:
!REPORT_GAS=true npx hardhat test test/OrderLifecycle.test.js